# Local test

Notebook này để chạy test local trên máy, để hiểu hơn về cahcs hoạt động của mô hình


In [25]:
from pathlib import Path
import inspect
import json
import re
import sys

import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()

BASE_MODEL_ID = "NlpHUST/gpt2-vietnamese"
LOCAL_MODEL_DIR = PROJECT_ROOT / "models" / "nlphust-gpt2-vietnamese"

# Ban tu sua file nay de doi cau hoi dau vao.
INPUT_PATH = PROJECT_ROOT / "scripts" / "input.json"
OUTPUT_PATH = PROJECT_ROOT / "scripts" / "local_base_model_output.json"

SAFE_EOS_ID = 50256
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 80
BATCH_SIZE = 1

# Doi thanh True neu muon tai lai model tu Hugging Face.
FORCE_PULL_MODEL = False

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOCAL_MODEL_DIR:", LOCAL_MODEL_DIR)
print("INPUT_PATH:", INPUT_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)
print({"device": DEVICE, "dtype": str(DTYPE)})

PROJECT_ROOT: c:\Users\Admin\Documents\1. UET\UET\Năm 3 - HKII\học sâu và ứng dụng\viet-gpt2-math-word-problems
LOCAL_MODEL_DIR: c:\Users\Admin\Documents\1. UET\UET\Năm 3 - HKII\học sâu và ứng dụng\viet-gpt2-math-word-problems\models\nlphust-gpt2-vietnamese
INPUT_PATH: c:\Users\Admin\Documents\1. UET\UET\Năm 3 - HKII\học sâu và ứng dụng\viet-gpt2-math-word-problems\scripts\input.json
OUTPUT_PATH: c:\Users\Admin\Documents\1. UET\UET\Năm 3 - HKII\học sâu và ứng dụng\viet-gpt2-math-word-problems\scripts\local_base_model_output.json
{'device': 'cpu', 'dtype': 'torch.float32'}


In [26]:
def pull_model_to_local(repo_id: str, local_dir: Path, force: bool = False):
    has_model = (local_dir / "config.json").exists()
    if has_model and not force:
        print("Model da co san tai local:", local_dir)
        return local_dir

    local_dir.mkdir(parents=True, exist_ok=True)
    kwargs = {
        "repo_id": repo_id,
        "local_dir": str(local_dir),
    }

    if "local_dir_use_symlinks" in inspect.signature(snapshot_download).parameters:
        kwargs["local_dir_use_symlinks"] = False

    print("Dang pull model tu Hugging Face:", repo_id)
    snapshot_download(**kwargs)
    print("Da pull xong model ve:", local_dir)
    return local_dir


MODEL_DIR = pull_model_to_local(BASE_MODEL_ID, LOCAL_MODEL_DIR, FORCE_PULL_MODEL)

Model da co san tai local: c:\Users\Admin\Documents\1. UET\UET\Năm 3 - HKII\học sâu và ứng dụng\viet-gpt2-math-word-problems\models\nlphust-gpt2-vietnamese


## Input file

Notebook đọc input từ `scripts/input.json`. PHẢI tự tạo thử file này trước khi chạy notebook.

File có thể là JSON array:

```json
[
  {
    "id": 0,
    "query_vi": "Lan có 12 viên kẹo. Lan cho Minh 5 viên rồi mẹ cho Lan thêm 8 viên. Hỏi Lan có bao nhiêu viên kẹo?",
    "type": "demo"
  }
]
```

Hoặc JSONL, mỗi dòng là một object JSON:

```jsonl
{"id": 0, "query_vi": "Câu hỏi thứ nhất...", "type": "demo"}
{"id": 1, "query_vi": "Câu hỏi thứ hai...", "type": "demo"}
```

`query_vi` là field bắt buộc. `id` và `type` là optional. `type` chỉ là nhãn phân loại, model không dùng field này để suy luận.

In [27]:
def load_records(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Chưa có file input: {path}")
    text = path.read_text(encoding="utf-8-sig").strip()
    if not text:
        return []
    if text[0] == "[":
        records = json.loads(text)
    else:
        records = [json.loads(line) for line in text.splitlines() if line.strip()]

    normalized = []
    for index, item in enumerate(records):
        if isinstance(item, str):
            item = {"query_vi": item}
        query_vi = item.get("query_vi") or item.get("question") or item.get("text")
        if not query_vi:
            raise ValueError(f"Record {index} khong co field query_vi/question/text: {item}")
        normalized.append({
            "id": item.get("id", index),
            "query_vi": query_vi,
            "type": item.get("type", ""),
        })
    return normalized


records = load_records(INPUT_PATH)
print("So cau hoi:", len(records))
records[:3]

So cau hoi: 2


[{'id': 0, 'query_vi': 'giải bài toán 1+1 bằng bao nhiêu?', 'type': 'GSM'},
 {'id': 1,
  'query_vi': 'Một lớp học có 24 học sinh. Mỗi bàn ngồi được 4 học sinh. Hỏi cần bao nhiêu bàn để đủ chỗ cho cả lớp?',
  'type': 'demo'}]

## Load model từ local


In [28]:
def load_causal_lm(model_dir: Path, dtype):
    try:
        return AutoModelForCausalLM.from_pretrained(
            str(model_dir),
            local_files_only=True,
            dtype=dtype,
        )
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(
            str(model_dir),
            local_files_only=True,
            torch_dtype=dtype,
        )


def set_safe_tokens(tokenizer, model):
    safe_eos_token = tokenizer.convert_ids_to_tokens(SAFE_EOS_ID)
    if safe_eos_token is not None:
        tokenizer.eos_token = safe_eos_token
        tokenizer.pad_token = safe_eos_token
    tokenizer.eos_token_id = SAFE_EOS_ID
    tokenizer.pad_token_id = SAFE_EOS_ID
    tokenizer.padding_side = "left"
    model.config.eos_token_id = SAFE_EOS_ID
    model.config.pad_token_id = SAFE_EOS_ID


tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    use_fast=True,
)
model = load_causal_lm(MODEL_DIR, DTYPE)
set_safe_tokens(tokenizer, model)
model.to(DEVICE)
model.eval()
torch.set_grad_enabled(False)

print("Loaded local model:", MODEL_DIR)
print({"pad_token_id": tokenizer.pad_token_id, "eos_token_id": tokenizer.eos_token_id})

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 18574.97it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: c:\Users\Admin\Documents\1. UET\UET\Năm 3 - HKII\học sâu và ứng dụng\viet-gpt2-math-word-problems\models\nlphust-gpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded local model: c:\Users\Admin\Documents\1. UET\UET\Năm 3 - HKII\học sâu và ứng dụng\viet-gpt2-math-word-problems\models\nlphust-gpt2-vietnamese
{'pad_token_id': 50256, 'eos_token_id': 50256}


## Generate output

In [29]:
def build_prompt(query_vi: str) -> str:
    return (
        "Hãy giải bài toán sau bằng tiếng Việt. "
        "Trình bày ngắn gọn và kết thúc bằng dòng 'Đáp án là: <số>'.\n"
        f"Bài toán: {query_vi}\n"
        "Lời giải:"
    )


def clean_model_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return "Lời giải: Đáp án là:"
    if not text.lower().startswith("lời giải"):
        text = "Lời giải: " + text
    return text


def generate_predictions(records):
    predictions = []
    for start in range(0, len(records), BATCH_SIZE):
        batch = records[start:start + BATCH_SIZE]
        prompts = [build_prompt(item["query_vi"]) for item in batch]
        encoded = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        ).to(DEVICE)

        generated = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.05,
            no_repeat_ngram_size=4,
            pad_token_id=SAFE_EOS_ID,
            eos_token_id=SAFE_EOS_ID,
        )

        prompt_width = encoded["input_ids"].shape[1]
        for offset, item in enumerate(batch):
            new_tokens = generated[offset][prompt_width:]
            model_output = clean_model_text(tokenizer.decode(new_tokens, skip_special_tokens=True))
            predictions.append({
                "id": item.get("id", start + offset),
                "query_vi": item["query_vi"],
                "type": item.get("type", ""),
                "model_output": model_output,
            })
    return predictions


predictions = generate_predictions(records)
predictions[:3]

[{'id': 0,
  'query_vi': 'giải bài toán 1+1 bằng bao nhiêu?',
  'type': 'GSM',
  'model_output': 'Lời giải: Bài toán 1+2 bằng bao nhiêu số? (Bài này có thể giải bằng tiếng Anh). Bài này có thể được giải bằng tiếng Việt, nhưng không được giải bằng chữ Latin. Bài toán 2+3 bằng bao nhiêu chữ? (Bài giải có thể giải theo tiếng Việt hoặc tiếng Anh). (Bài này không thể giải bằng chữ La-tinh). Bài toán 3+4'},
 {'id': 1,
  'query_vi': 'Một lớp học có 24 học sinh. Mỗi bàn ngồi được 4 học sinh. Hỏi cần bao nhiêu bàn để đủ chỗ cho cả lớp?',
  'type': 'demo',
  'model_output': 'Lời giải: Số học sinh trong lớp là: (1+2+3+4+5+6+7+8+9+10+11+12+13+14+15+16+17+18+19+19+20+21+22+23+24+25+26+27+28+29+30+31+32+33+34+35+'}]

## Save output

In [30]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved output:", OUTPUT_PATH)
print(json.dumps(predictions[:3], ensure_ascii=False, indent=2))

Saved output: c:\Users\Admin\Documents\1. UET\UET\Năm 3 - HKII\học sâu và ứng dụng\viet-gpt2-math-word-problems\scripts\local_base_model_output.json
[
  {
    "id": 0,
    "query_vi": "giải bài toán 1+1 bằng bao nhiêu?",
    "type": "GSM",
    "model_output": "Lời giải: Bài toán 1+2 bằng bao nhiêu số? (Bài này có thể giải bằng tiếng Anh). Bài này có thể được giải bằng tiếng Việt, nhưng không được giải bằng chữ Latin. Bài toán 2+3 bằng bao nhiêu chữ? (Bài giải có thể giải theo tiếng Việt hoặc tiếng Anh). (Bài này không thể giải bằng chữ La-tinh). Bài toán 3+4"
  },
  {
    "id": 1,
    "query_vi": "Một lớp học có 24 học sinh. Mỗi bàn ngồi được 4 học sinh. Hỏi cần bao nhiêu bàn để đủ chỗ cho cả lớp?",
    "type": "demo",
    "model_output": "Lời giải: Số học sinh trong lớp là: (1+2+3+4+5+6+7+8+9+10+11+12+13+14+15+16+17+18+19+19+20+21+22+23+24+25+26+27+28+29+30+31+32+33+34+35+"
  }
]
